In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast

In [ ]:
ratings = pd.read_csv('../data/preprocessed/ratings_cleaned.csv')
movies = pd.read_csv('../data/preprocessed/movies_cleaned.csv')

In [ ]:
# Convert the text back into lists for the 'genres' column
# ast.literal_eval safely converts the string "['A', 'B']" into an actual list ['A', 'B']
movies['genres'] = movies['genres'].apply(ast.literal_eval)

print(f"Data loaded! Ratings: {len(ratings)}, Movies: {len(movies)}")

In [ ]:
movies.sample(3)

In [ ]:
ratings.sample(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(ax=ax[0], data=ratings, x='rating',  color = 'r')
ax[0].set_title('Statistical Spread', fontsize=14)
ax[0].set_xlabel("Movie Rating")
ax[0].set_ylabel("Number of Ratings")

sns.countplot(ax=ax[1], data=ratings, x='rating', color = 'r', edgecolor='black', linewidth=0.5)
ax[1].set_title('Frequency of Each Rating', fontsize=14)
ax[1].set_xlabel("Movie Rating")
ax[1].set_ylabel("Number of Ratings")

fig.suptitle('Distribution of User Ratings', fontsize=18, fontweight='bold')

plt.tight_layout()

In [ ]:
ratings_per_user = ratings.groupby('userId').size().reset_index(name='num_ratings').sort_values(by='num_ratings', ascending=False)

In [ ]:
print("Top 10 most active users")
ratings_per_user.head(10)

In [ ]:
ratings_per_user['num_ratings'].describe()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(data=ratings_per_user, x='num_ratings', bins=200, color='r', edgecolor='black', linewidth=0.5)

plt.xlim(0, 500)

plt.title('Distribution of Number of Ratings per User (Zoomed In 0-500)', fontsize=14, weight='bold')
plt.xlabel('Number of Ratings Given')
plt.ylabel('Number of Users')

plt.tight_layout()

In [ ]:
avg_user_rating = ratings.groupby('userId')['rating'].mean().reset_index(name='avg_rating')

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(data=avg_user_rating, x='avg_rating', bins=30, color='r', edgecolor='black', linewidth=0.5)

plt.title('Distribution of Average Ratings per User', fontsize=14, weight='bold')
plt.xlabel('Average Rating Given')
plt.ylabel('Number of Users')

plt.tight_layout()

In [ ]:
genre_counts = {}

for x in movies['genres']:
    for y in x:
        if y in genre_counts:
            genre_counts[y] += 1
        else:
            genre_counts[y] = 1

In [ ]:
count_per_genre = (
    pd.DataFrame.from_dict(genre_counts, orient='index')
    .reset_index()
    .sort_values(by=0, ascending=False)
    .rename(columns={"index" : "genre", 0: "count"})
)

In [ ]:
count_per_genre.head(10)

In [ ]:
print(f"We have {len(count_per_genre)} unique genres.")

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(data=count_per_genre.head(10), x='genre', y='count', color='r', edgecolor='black', linewidth=0.5)

plt.title("Top 10 Most Frequent Movie Genres", fontsize=14, weight='bold')
plt.xlabel("Movie Genre")
plt.ylabel("Number of Movies")

plt.tight_layout()

In [ ]:
movies_exploded = movies.explode('genres')

In [ ]:
genre_ratings = ratings.merge(movies_exploded, on='movieId')

In [ ]:
genre_stats = genre_ratings.groupby('genres').agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

In [ ]:
top_rated_genres = genre_stats.sort_values(by='avg_rating', ascending=False).head(10)

In [ ]:
top_rated_genres

In [ ]:
plt.figure(figsize=(12, 8))

ax = sns.barplot(
    data=top_rated_genres, 
    y='genres', 
    x='avg_rating', 
    color='r', 
    edgecolor='black', 
    linewidth=0.5
)

ax.bar_label(ax.containers[0], fmt='%.2f', padding=5, fontsize=10)

plt.title('Highest Rated Movie Genres (Average User Score)', fontsize=14, weight='bold')
plt.xlabel('Average Rating (out of 5.0)')
plt.ylabel('Genre')

plt.tight_layout()

In [ ]:
ratings_per_film = ratings.groupby('movieId').size().reset_index(name='num_ratings').sort_values(by='num_ratings', ascending=False)

In [ ]:
ratings_per_film

Note: Due to filtering inactive users after unpopular films, the minimum ratings per item slightly dropped from 5 to 4, which is acceptable for the model.

In [ ]:
top_10_films = ratings_per_film.head(10).merge(movies, on='movieId', how='left')

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(data=top_10_films, y='title', x='num_ratings', color='r', edgecolor='black', linewidth=0.5)

plt.title('Top 10 Most Rated Movies', fontsize=14, weight='bold')
plt.xlabel('Number of Ratings')
plt.ylabel('')

plt.tight_layout()

## Sparsity & Density

In [ ]:
n_users = ratings['userId'].nunique()

In [ ]:
n_movies = movies['movieId'].nunique()

In [ ]:
n_interactions = len(ratings)

In [ ]:
print(f"We have {n_users} unique users and {n_movies} unique films. Total number of ratings: {n_interactions}")

In [ ]:
total_possible = n_movies * n_users # Maximum possible number of ratings

In [ ]:
density = np.round((n_interactions/total_possible)*100, 2) # % of filled cells in user-item matrix

In [ ]:
sparsity = np.round((1-(n_interactions/total_possible))*100, 2) # % of empty cells in user-item matrix

In [ ]:
print(f"Matrix Density (% of filled cells): {density}%")